# Modular NLP Pipeline Project

### Tools: Python, scikit-learn, NLTK

This project refactors a text-preprocessing and similarity workflow into reusable classes.

The final architecture contains three main components:

1. `PreprocessingModule` — cleans raw text.
2. `VectorizerModule` — learns TF-IDF features and calculates similarity.
3. `Pipeline` — connects the modules and exposes one simple `run(query, corpus)` method.

The notebook also includes:

- a 15-document corpus
- five query tests with ranked similarity scores
- three edge-case tests
- a data-flow architecture diagram
- a design decision note explaining modularity
- explanations of every important new command

## 1. What are we changing from the earlier projects?

In the earlier projects, preprocessing, vectorisation, similarity, and printing were written as a sequence of separate notebook cells.

That approach is fine for learning, but as a project grows it can become difficult to maintain.

For example:

```text
Raw text
   ↓
preprocessing code
   ↓
TF-IDF code
   ↓
similarity code
   ↓
ranking code
   ↓
printing code
```

If everything is inside one large script, changing one part can accidentally break another part.

Here we separate the responsibilities:

```text
PreprocessingModule
        ↓
VectorizerModule
        ↓
     Pipeline
        ↓
 ranked results
```

Each class has a focused job.

## 2. Import the libraries

In [ ]:
import re
import numpy as np

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### New commands explained

- `import re` loads Python's regular-expression module. We use it to remove unwanted characters.
- `import numpy as np` imports NumPy and gives it the short name `np`. NumPy is useful for arrays, scores, and ranking indices.
- `word_tokenize` splits text into tokens.
- `stopwords` gives us NLTK's built-in English stopword list.
- `PorterStemmer` reduces words to stems.
- `TfidfVectorizer` converts cleaned text into numerical TF-IDF vectors.
- `cosine_similarity` compares the directions of the resulting vectors.

The aliases `np` and the imported class/function names are conventional Python shortcuts.

## 3. Prepare the NLTK resources

NLTK's algorithms are installed with the library, but some language resources are downloaded separately.

In [ ]:
import nltk

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

### Why do we download these resources?

- `punkt` and `punkt_tab` support NLTK tokenisation.
- `stopwords` supplies the English stopword corpus.
- `quiet=True` reduces unnecessary download messages in the notebook.

If the resources are already installed, NLTK will simply confirm that they are available.

## 4. Build the `PreprocessingModule` class

### Responsibility

The `PreprocessingModule` is responsible **only for preparing text**.

Its `transform(text)` method will:

1. validate the input
2. convert it to lowercase
3. tokenize it
4. remove stopwords
5. remove punctuation/symbols
6. remove empty tokens
7. apply Porter stemming
8. return the cleaned text

This is an example of **single responsibility**: this class should not calculate TF-IDF or similarity.

In [ ]:
class PreprocessingModule:
    """Clean raw English text before vectorisation."""

    def __init__(self, remove_stopwords=True, apply_stemming=True):
        """
        Parameters
        ----------
        remove_stopwords : bool
            Whether common English stopwords should be removed.
        apply_stemming : bool
            Whether Porter stemming should be applied.
        """
        self.remove_stopwords = remove_stopwords
        self.apply_stemming = apply_stemming
        self.stop_words = set(stopwords.words("english"))
        self.stemmer = PorterStemmer()

    def transform(self, text):
        """Return a cleaned string suitable for vectorisation."""

        if not isinstance(text, str):
            raise TypeError("Input text must be a string.")

        if not text.strip():
            raise ValueError("Input cannot be an empty string.")

        if len(text.strip()) == 1:
            raise ValueError("Single-character queries are not supported.")

        tokens = word_tokenize(text.lower())

        if self.remove_stopwords:
            tokens = [
                token for token in tokens
                if token not in self.stop_words
            ]

        tokens = [
            re.sub(r"[^a-zA-Z]", "", token)
            for token in tokens
        ]

        tokens = [token for token in tokens if token]

        if not tokens:
            raise ValueError(
                "Input contains no usable alphabetic words."
            )

        if self.apply_stemming:
            tokens = [
                self.stemmer.stem(token)
                for token in tokens
            ]

        return " ".join(tokens)

## 5. Understand the `PreprocessingModule`

There are several new Python ideas here.

### `class`

```python
class PreprocessingModule:
```

creates a reusable blueprint. Instead of writing the same preprocessing commands repeatedly, we create one object and call its methods.

### `__init__`

```python
def __init__(self, remove_stopwords=True, apply_stemming=True):
```

runs automatically when we create the object.

For example:

```python
preprocessor = PreprocessingModule()
```

creates an instance with stopword removal and stemming enabled.

### `self`

`self` refers to the current object.

For example:

```python
self.stop_words
```

means that the stopword set belongs to this particular `PreprocessingModule` object.

### Type checking

```python
isinstance(text, str)
```

checks whether the input is actually a Python string.

### `raise ValueError`

```python
raise ValueError("...")
```

stops the operation and gives the user a meaningful error message instead of allowing a confusing failure later.

### List comprehensions

Code such as:

```python
[token for token in tokens if token not in self.stop_words]
```

creates a new list containing only tokens that pass the condition.

### Regex

```python
re.sub(r"[^a-zA-Z]", "", token)
```

keeps alphabetic English characters and removes punctuation, numbers, emojis, and other symbols.

This is deliberately strict because our TF-IDF demonstration is based on alphabetic English words.

## 6. Test the preprocessing module

Before connecting it to the rest of the pipeline, we should test the module by itself.

This is one of the advantages of modular design: each component can be tested independently.

In [ ]:
preprocessor = PreprocessingModule()

example_text = "WOW!!! The dogs are running quickly 😊."

print("Raw text       :", example_text)
print("Processed text :", preprocessor.transform(example_text))

### Why this test matters

If preprocessing produces an unexpected result, we know where to investigate: the `PreprocessingModule`.

In a monolithic script, the problem could be hidden somewhere among preprocessing, vectorisation, similarity, and output code.

## 7. Build the `VectorizerModule` class

### Responsibility

The `VectorizerModule` handles the numerical side of the project.

It will:

- fit a TF-IDF vocabulary on a corpus
- transform a query into that same vector space
- calculate similarity scores
- rank corpus documents

It does **not** clean raw text. That belongs to `PreprocessingModule`.

In [ ]:
class VectorizerModule:
    """Fit TF-IDF vectors and compare a query with a corpus."""

    def __init__(self):
        self.vectorizer = TfidfVectorizer()
        self.corpus_vectors = None
        self.corpus = None

    def fit(self, corpus):
        """Learn TF-IDF features from the supplied processed corpus."""

        if not corpus:
            raise ValueError("Corpus cannot be empty.")

        if any(not isinstance(doc, str) for doc in corpus):
            raise TypeError("Every corpus document must be a string.")

        if any(not doc.strip() for doc in corpus):
            raise ValueError("Corpus contains an empty document.")

        self.corpus = corpus
        self.corpus_vectors = self.vectorizer.fit_transform(corpus)

        return self

    def transform(self, query):
        """Transform a processed query into the fitted TF-IDF space."""

        if self.corpus_vectors is None:
            raise RuntimeError("Call fit(corpus) before transform(query).")

        if not isinstance(query, str):
            raise TypeError("Query must be a string.")

        if not query.strip():
            raise ValueError("Query cannot be empty.")

        return self.vectorizer.transform([query])

    def similarity(self, query_vector):
        """Return cosine similarity between query and every corpus document."""

        if self.corpus_vectors is None:
            raise RuntimeError("Call fit(corpus) before similarity().")

        scores = cosine_similarity(
            query_vector,
            self.corpus_vectors
        )[0]

        return scores

    def rank(self, query_vector):
        """Return corpus indices sorted from most to least similar."""

        scores = self.similarity(query_vector)
        return np.argsort(scores)[::-1]

## 8. Understand `VectorizerModule`

### `self.vectorizer`

The TF-IDF vectorizer is stored inside the object so that the same vocabulary is reused later.

This is important. If we fitted a new vectorizer for every query, the query and corpus might use different feature dimensions.

### `fit(corpus)`

`fit()` learns the vocabulary and TF-IDF statistics from the corpus.

### `transform(query)`

`transform()` uses the vocabulary already learned during `fit()`.

### `similarity(query_vector)`

This calculates the query-to-corpus cosine similarity scores.

### `rank(query_vector)`

```python
np.argsort(scores)[::-1]
```

first finds the indices that would sort the scores, then reverses them so the largest score appears first.

The important design principle is:

> `fit()` learns; `transform()` reuses what was learned.

## 9. Build the `Pipeline` class

### Responsibility

The `Pipeline` is the coordinator.

The user should not need to know every internal step. Instead, the public interface is simply:

```python
pipeline.run(query, corpus)
```

Internally it will:

1. preprocess every corpus document
2. preprocess the query
3. fit TF-IDF on the processed corpus
4. transform the processed query
5. calculate similarity
6. rank the results
7. return the ranked documents and scores

In [ ]:
class Pipeline:
    """Connect preprocessing and vectorisation into one searchable pipeline."""

    def __init__(self, preprocessor=None, vectorizer=None):
        self.preprocessor = preprocessor or PreprocessingModule()
        self.vectorizer = vectorizer or VectorizerModule()

    def run(self, query, corpus):
        """Preprocess, vectorise, compare, and rank a query against a corpus."""

        if not isinstance(corpus, list):
            raise TypeError("Corpus must be a list of strings.")

        if not corpus:
            raise ValueError("Corpus cannot be empty.")

        cleaned_corpus = [
            self.preprocessor.transform(document)
            for document in corpus
        ]

        cleaned_query = self.preprocessor.transform(query)

        self.vectorizer.fit(cleaned_corpus)
        query_vector = self.vectorizer.transform(cleaned_query)
        scores = self.vectorizer.similarity(query_vector)
        ranked_indices = self.vectorizer.rank(query_vector)

        ranked_results = [
            {
                "rank": rank,
                "document_index": int(index),
                "document": corpus[index],
                "processed_document": cleaned_corpus[index],
                "similarity": float(scores[index])
            }
            for rank, index in enumerate(ranked_indices, start=1)
        ]

        return ranked_results

## 10. Understand the `Pipeline` class

### `or`

```python
preprocessor or PreprocessingModule()
```

means:

- use the object supplied by the user if one was supplied
- otherwise create a default preprocessor

This makes the class flexible and testable.

### Why return dictionaries?

Each result contains:

- rank
- original document index
- original document
- processed document
- similarity score

A dictionary makes the result self-explanatory and easy to access.

For example:

```python
result["similarity"]
```

gets the score.

### Why keep the original document?

The model works on processed text, but the user needs to see the original document in the final ranked output. Keeping both makes the system easier to inspect and debug.

## 11. Create the 15-document corpus

The corpus contains five documents from each of three broad topics:

- **Animals/pets**
- **Travel**
- **Technology**

The overlap within each topic gives the similarity model enough vocabulary to produce meaningful rankings.

In [ ]:
corpus = [
    "Dogs are loyal pets that enjoy long walks with their owners.",
    "Puppies need gentle training, healthy food, and regular exercise.",
    "Cats are quiet pets that enjoy comfortable homes and gentle care.",
    "Veterinarians recommend regular health checks for dogs and cats.",
    "Animal shelters help people adopt friendly pets and care for them.",

    "Travelers often visit beautiful cities during summer holidays.",
    "Tourists book hotels before traveling to popular destinations.",
    "A train journey can take travelers through beautiful countryside.",
    "Travel planning includes choosing flights, hotels, and local activities.",
    "Tourists enjoy exploring museums, restaurants, parks, and historic cities.",

    "Modern smartphones use powerful processors and long-lasting batteries.",
    "A laptop needs enough memory and a fast processor for software development.",
    "Computer software can improve productivity and protect digital data.",
    "Cloud computing allows applications and files to be accessed online.",
    "Cybersecurity helps protect computer networks from digital attacks."
]

print("Number of documents:", len(corpus))

for index, document in enumerate(corpus, start=1):
    print(f"{index:02d}. {document}")

### New commands explained

- `enumerate(corpus, start=1)` begins counting at 1 instead of Python's default 0.
- `f"{index:02d}"` formats the number with two digits, so we get `01`, `02`, ..., `15`.
- `len(corpus)` tells us how many documents are in the corpus.

We now have exactly the **15 documents required by the assignment**.

## 12. Create and run the complete pipeline

Now the three modules are connected.

In [ ]:
pipeline = Pipeline()

query = "I need a friendly dog that enjoys exercise and walks."

results = pipeline.run(query, corpus)

for result in results[:5]:
    print(
        f"{result['rank']}. "
        f"Score={result['similarity']:.3f} | "
        f"{result['document']}"
    )

### What happened?

One call:

```python
pipeline.run(query, corpus)
```

performed the entire workflow.

This is the main benefit of the architecture. A user does not have to manually call preprocessing, TF-IDF fitting, query transformation, cosine similarity, and ranking.

The pipeline hides the implementation details while keeping the individual modules independently testable.

## 13. Test five different queries

The assignment requires five query tests on the 15-document corpus.

We will test:

1. dog/pet topic
2. travel topic
3. technology topic
4. cybersecurity topic
5. cat/pet topic

For each query, the top five ranked results and similarity scores are printed.

In [ ]:
queries = {
    "Q1 - Dog and exercise": "A dog needs regular exercise and enjoys walking with its owner.",
    "Q2 - Travel planning": "I want to plan a trip, book a hotel, and visit a beautiful city.",
    "Q3 - Laptop technology": "I need a fast computer with enough memory for software development.",
    "Q4 - Cybersecurity": "How can I protect my computer network from digital attacks?",
    "Q5 - Cat care": "Cats need gentle care, comfortable homes, and good health."
}

for query_name, query in queries.items():
    print("=" * 90)
    print(query_name)
    print("Query:", query)
    print("=" * 90)

    results = pipeline.run(query, corpus)

    for result in results[:5]:
        print(
            f"{result['rank']}. "
            f"score={result['similarity']:.3f} | "
            f"{result['document']}"
        )

    print()

### Why do we print the scores?

The ranked sentence alone is not enough. The similarity score tells us **how strongly** the model matched the query with each document.

A result with `0.60` is a stronger lexical match than one with `0.10`, although the exact interpretation depends on the corpus and task.

The five tests also let us see whether the pipeline behaves consistently across different topics.

## 14. Edge-case handling

The assignment specifically requires three edge cases:

1. empty string
2. single-character query
3. input containing only numbers or symbols

A good pipeline should fail **clearly and early** instead of producing confusing downstream errors.

In [ ]:
edge_cases = {
    "Empty string": "",
    "Single character": "x",
    "Only numbers": "123456789",
    "Only symbols": "@#$%^&*!"
}

for case_name, bad_query in edge_cases.items():
    print("=" * 70)
    print(case_name)
    print("Input:", repr(bad_query))

    try:
        pipeline.run(bad_query, corpus)
        print("Result: unexpectedly accepted")
    except (ValueError, TypeError, RuntimeError) as error:
        print("Handled safely:", error)

### New error-handling commands explained

#### `try`

```python
try:
    ...
```

means: attempt to execute this code.

#### `except`

```python
except ValueError as error:
```

catches a known error instead of allowing the notebook to stop completely.

#### `repr()`

```python
repr(bad_query)
```

shows the exact representation of the input. This is useful for empty strings because `repr("")` visibly shows `''`.

### Why handle these cases?

Without validation:

- an empty query may create an invalid TF-IDF input
- a one-character query may be meaningless for the intended search task
- numbers/symbols may disappear during preprocessing, leaving no usable vocabulary

Early validation produces a clear error message and makes the pipeline safer.

## 15. Test the modules independently

A modular design is useful because each component can be tested separately.

First, test preprocessing.

In [ ]:
test_text = "The friendly dogs are running quickly!"

print("Original :", test_text)
print("Processed:", preprocessor.transform(test_text))

Now test the vectorizer independently.

This demonstrates that the vectorizer can work without the `Pipeline` wrapper when necessary.

In [ ]:
small_corpus = [
    "friendly dog walks",
    "friendly cat home",
    "computer software"
]

small_preprocessor = PreprocessingModule()
small_vectorizer = VectorizerModule()

cleaned_small = [
    small_preprocessor.transform(text)
    for text in small_corpus
]

small_vectorizer.fit(cleaned_small)

query_vector = small_vectorizer.transform(
    small_preprocessor.transform("friendly dog")
)

scores = small_vectorizer.similarity(query_vector)

print("Scores:", np.round(scores, 3))

This is an important software-engineering advantage: if the pipeline produces a wrong result, we can determine whether the problem is in preprocessing, vectorisation, or orchestration.

## 16. Data-flow architecture diagram

The following diagram shows how raw text travels through the system.

The key idea is that each module has one main responsibility.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(14, 5))
ax.set_xlim(0, 14)
ax.set_ylim(0, 5)
ax.axis("off")

boxes = [
    (0.5, 1.7, 2.0, 1.2, "Raw Input\nQuery + Corpus"),
    (3.2, 1.7, 2.4, 1.2, "PreprocessingModule\ntransform(text)"),
    (6.4, 1.7, 2.4, 1.2, "VectorizerModule\nfit() + transform()"),
    (9.6, 1.7, 2.4, 1.2, "Cosine Similarity\n+ Ranking"),
    (12.2, 1.7, 1.4, 1.2, "Ranked\nOutput")
]

for x, y, width, height, label in boxes:
    box = FancyBboxPatch(
        (x, y),
        width,
        height,
        boxstyle="round,pad=0.03"
    )
    ax.add_patch(box)
    ax.text(
        x + width / 2,
        y + height / 2,
        label,
        ha="center",
        va="center",
        fontsize=10
    )

for start_x, end_x in [
    (2.5, 3.2),
    (5.6, 6.4),
    (8.8, 9.6),
    (12.0, 12.2)
]:
    arrow = FancyArrowPatch(
        (start_x, 2.3),
        (end_x, 2.3),
        arrowstyle="->",
        mutation_scale=15
    )
    ax.add_patch(arrow)

ax.text(
    7,
    4.1,
    "Modular NLP Semantic Search Data Flow",
    ha="center",
    fontsize=15
)

plt.tight_layout()
plt.show()

### Diagram commands explained

- `plt.subplots()` creates the figure and drawing area.
- `ax.set_xlim()` and `ax.set_ylim()` define the drawing coordinates.
- `ax.axis("off")` hides ordinary chart axes because this is an architecture diagram rather than a graph.
- `FancyBboxPatch` creates rounded boxes.
- `ax.add_patch()` places a shape on the diagram.
- `ax.text()` writes labels.
- `FancyArrowPatch` creates arrows showing the direction of data flow.
- `plt.show()` displays the completed diagram.

The architecture is:

```text
Raw Input
   ↓
PreprocessingModule
   ↓
VectorizerModule
   ↓
Cosine Similarity + Ranking
   ↓
Final Ranked Output
```

## 17. Design decision note: why modularity is better than one large script

### Single-responsibility modules

A monolithic script might contain hundreds of lines that clean text, fit a vectorizer, calculate similarity, rank results, validate input, and print output.

That creates several problems:

- **Harder debugging:** an error can come from any stage.
- **Harder testing:** individual parts cannot easily be tested in isolation.
- **Harder extension:** changing preprocessing can accidentally affect similarity code.
- **Code duplication:** the same preprocessing logic may be copied into several places.
- **Poor readability:** the main workflow is buried inside implementation details.

Our modular design separates the concerns:

| Component | Responsibility |
|---|---|
| `PreprocessingModule` | Clean raw text |
| `VectorizerModule` | TF-IDF and similarity |
| `Pipeline` | Connect the modules |
| Test code | Demonstrate behaviour |

### Why this is easier to debug

Suppose a query receives an unexpectedly low similarity score.

We can test:

```text
PreprocessingModule → Did important words disappear?
        ↓
VectorizerModule   → Was the vocabulary fitted correctly?
        ↓
Similarity          → Are the scores calculated correctly?
        ↓
Pipeline            → Were the modules connected correctly?
```

Each stage has a smaller responsibility, so the source of the problem is easier to locate.

### Why this is easier to extend

Later, we could replace:

```python
PorterStemmer()
```

with another preprocessing strategy without rewriting the ranking code.

Likewise, we could replace TF-IDF with a modern embedding model while keeping the same high-level pipeline interface.

That is the main reason modular components are easier to maintain and extend than a monolithic script.

## 18. Final project summary

The final architecture follows the principle of **separation of concerns**:

```text
                    ┌────────────────────────┐
                    │     Raw Query/Corpus   │
                    └───────────┬────────────┘
                                ↓
                    ┌────────────────────────┐
                    │  PreprocessingModule   │
                    │      transform()       │
                    └───────────┬────────────┘
                                ↓
                    ┌────────────────────────┐
                    │    VectorizerModule    │
                    │ fit() + transform()    │
                    └───────────┬────────────┘
                                ↓
                    ┌────────────────────────┐
                    │ Cosine Similarity      │
                    │ + Ranking              │
                    └───────────┬────────────┘
                                ↓
                    ┌────────────────────────┐
                    │   Ranked Results        │
                    │  document + score      │
                    └────────────────────────┘
```

The user-facing entry point is:

```python
pipeline.run(query, corpus)
```

The pipeline hides the internal details while the classes remain independently testable.

### Submission checklist

- [ ] `PreprocessingModule` class created with `transform(text)`.
- [ ] Documented preprocessing parameters included.
- [ ] `VectorizerModule` class created with `fit(corpus)` and `transform(query)`.
- [ ] `Pipeline` class created.
- [ ] `Pipeline.run(query, corpus)` returns ranked results with similarity scores.
- [ ] 15-document corpus included.
- [ ] Five query tests included.
- [ ] Three required edge cases tested.
- [ ] Architecture/data-flow diagram included.
- [ ] Design decision note included.
- [ ] New Python/NLTK/scikit-learn commands explained throughout.